# Arabizi keyboard: Llama 3.2 1B v4 (synthetic rows aimed at v3's gaps, round 5 in real-chat style)

**Name:** this notebook trained the shipped model on 15 Nov 2026 under the tag `v5` (run `20261115-160254-llama3.2-1b-lora-v5`); it was renamed v4 when the interim run was skipped (ADR 0038). Rerunning it writes files named v4.

One LoRA run, `v5`: the same settings as `v3` and `v4` (rank 32, alpha 64, lr 2e-4, 3 epochs) on the v5 export: the 18,039 v3 training rows plus 39,340 synthetic rows checked by Moustafa: all of v4's (25,169) and 14,171 from round 5, written to look like real phone chat (longer messages, emoji, dropped vowels, ط ص ض ظ typed as t s d z) and aimed at v4's gaps (words read from context, heavy letters, English and French kept in Latin, new word forms, rare words). 122,492 examples, AI-derived share 67%.

**How to run:** upload `arabizi-train-v5` as a **new dataset** (or a new version of `arabizi-train`). Then Settings → Accelerator → **GPU T4** · **Internet on** · Add Input → that dataset (latest version). Then **Save Version → Save & Run All (Commit)**: about 7 to 8 hours in the background (under Kaggle's 12-hour limit); the outputs stay on the version page. For about 5 hours instead, change `"3"` epochs below by adding `"--epochs", "2"` (v3 and v4 were both best at epoch 2).

**Data:** dev and test are never in the export. v5 is scored on the laptop against v3 and v4: both real dev sets and the synthetic test set (v4's split is frozen, so the comparison is fair).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as the shipped model


In [ ]:
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# Same settings as v3 and v4 (rank 32, alpha 64, lr 2e-4, 3 epochs). For 2 epochs: ["--rank", "32", "--alpha", "64", "--epochs", "2"].
RUNS = {
    "v4": ["--rank", "32", "--alpha", "64"],
}

for tag, extra in RUNS.items():
    print(f"\n=== {tag}: {' '.join(extra)}", flush=True)
    # check=True: a failed run stops the notebook instead of exporting nothing.
    subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", "llama3.2-1b", "--data", DATA,
                    "--out", OUT, "--tag", tag, *extra], check=True)
    run = max(glob.glob(f"{OUT}/*-llama3.2-1b-lora-{tag}"))
    subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", f"arabizi-llama3.2-1b-{tag}", run],
                   check=True)
    subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)


## Results and download

The table shows each run's best validation loss (training-style rows: **not** the decision; accuracy on the Mac decides). From the version's **Output** tab, use **Download all** (one zip) and put it in `~/Downloads`; then score the runs on the Mac (docs/TRAINING.md).


In [ ]:
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min)")
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"
